In [1]:
import pandas as pd
import numpy as np

# 1. Clean the RVD Housing Data (1.4Q.csv)
df_rvd = pd.read_csv('data/1.4Q.csv', skiprows=1)
df_rvd = df_rvd[['Quarter', 'All Classes']].copy()

df_rvd['Year'] = df_rvd['Quarter'].str.split('/').str[1]
df_rvd['Start_Month'] = df_rvd['Quarter'].str.split('-').str[0]

df_rvd['Date'] = pd.to_datetime(df_rvd['Year'] + '-' + df_rvd['Start_Month']).dt.to_period('Q')
df_rvd = df_rvd[['Date', 'All Classes']].rename(columns={'All Classes': 'Housing_Index'})
df_rvd['Housing_Index'] = pd.to_numeric(df_rvd['Housing_Index'], errors='coerce')


# 2. Clean the CPI Data (Table 510-60001_en.csv)
df_cpi = pd.read_csv('data/Table 510-60001_en.csv', skiprows=4)
df_cpi.rename(columns={
    df_cpi.columns[0]: 'Year',
    df_cpi.columns[1]: 'Month',
    df_cpi.columns[2]: 'CPI_Index'
}, inplace=True)

df_cpi = df_cpi.dropna(subset=['Month']).copy()

df_cpi['Month'] = df_cpi['Month'].str.strip()
df_cpi['Year'] = df_cpi['Year'].astype(str).str.strip()

df_cpi['Date'] = pd.to_datetime(df_cpi['Year'] + '-' + df_cpi['Month'], format='%Y-%b').dt.to_period('M')
df_cpi['CPI_Index'] = pd.to_numeric(df_cpi['CPI_Index'], errors='coerce')

# Convert Monthly CPI to Quarterly to match housing data
df_cpi['Date'] = df_cpi['Date'].dt.asfreq('Q')
df_cpi_q = df_cpi.groupby('Date')['CPI_Index'].mean().reset_index()


# 3. Merge them together!
final_dataset = pd.merge(df_rvd, df_cpi_q, on='Date', how='inner')

print(final_dataset)


       Date  Housing_Index   CPI_Index
0    1982Q1           22.5   22.933333
1    1982Q2           21.7   23.533333
2    1982Q3           20.6   24.033333
3    1982Q4           19.5   24.566667
4    1983Q1           18.5   25.133333
..      ...            ...         ...
172  2025Q1          285.9  108.633333
173  2025Q2          286.6  108.333333
174  2025Q3          290.1  109.000000
175  2025Q4          297.7  109.533333
176  2026Q1          308.1  110.333333

[177 rows x 3 columns]


In [2]:
!pip install xgboost scikit-learn

In [3]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Define your Features (X) and Target (y)
# We are predicting Housing_Index using CPI_Index 
X = final_dataset[['CPI_Index']]
y = final_dataset['Housing_Index']

# 2. Split the data (80% for training the model, 20% for testing its accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize the XGBoost Regressor
# n_estimators is the number of 'trees', learning_rate controls how aggressively it learns
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1)

# 4. Train the Model!
model.fit(X_train, y_train)

# 5. Make predictions on the unseen test data
predictions = model.predict(X_test)

# 6. Evaluate the results
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("✅ Model Training Complete!")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2) Score: {r2:.4f}")

✅ Model Training Complete!
Mean Absolute Error (MAE): 15.92
R-squared (R2) Score: 0.9518


In [4]:
import pandas as pd

# 1. Create a tiny dataset with your estimated future CPI
# Let's say economists predict CPI will be 112.5 later this year
future_data = pd.DataFrame({'CPI_Index': [112.5]})

# 2. Ask your trained XGBoost model to predict the Housing Index
future_prediction = model.predict(future_data)

# 3. Print the result
print(f"If CPI hits 112.5, the predicted HK Housing Index is: {future_prediction[0]:.2f}")

If CPI hits 112.5, the predicted HK Housing Index is: 306.10
